In [1]:
from datasets import load_dataset
dataset = load_dataset("Exploration-Lab/IL-TUR", "pcr")

In [2]:
dataset

DatasetDict({
    train_candidates: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 4320
    })
    dev_candidates: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 1023
    })
    test_candidates: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 1727
    })
    train_queries: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 827
    })
    dev_queries: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 118
    })
    test_queries: Dataset({
        features: ['id', 'text', 'relevant_candidates'],
        num_rows: 237
    })
})

## RR-Augmented Prior Case Retrieval

**Why naive BERT fails:** Legal cases are 5,000–20,000+ words. The top-512-token window captures only the procedural header (party names, court, dates), never reaching the legally meaningful content deeper in the document. This explains the paradox where domain-adapted models (LegalBERT, InLegalBERT) perform *worse* than vanilla BERT — they overfit to legal boilerplate in the preamble.

**Strategy — RR-filtered Bi-Encoder:**
1. Use the SOTA RR model to label every sentence as: `Fact`, `Argument`, `Statute`, `RatioOfTheDecision`, `Precedent`, `RulingByLowerCourt`, `RulingByPresentCourt`
2. Filter each document to keep only **high-signal** labels: `RatioOfTheDecision`, `Statute`, `Precedent`, `Argument`
3. Feed this filtered text to the bi-encoder — the 512-token window now covers the core legal reasoning instead of preamble boilerplate

**Step 1 below:** Prepare the IL-TUR dataset in the `.sav` format expected by `rr_pipeline.py`, then run the pipeline to get per-sentence RR labels for all splits.

In [3]:
import pickle as pkl
import os

# The rr_pipeline.py expects .sav files under its Input_Data/ directory.
# Each .sav file is a dict: {"candidate_data": {case_id: [sentence1, sentence2, ...]}}

RR_PIPELINE_DIR = "../others/Rhetorical-Roles/Code/RR_pipeline"
rr_input_path = os.path.join(RR_PIPELINE_DIR, "Input_Data")
os.makedirs(rr_input_path, exist_ok=True)

splits = [
    "train_queries", "dev_queries", "test_queries",
    "train_candidates", "dev_candidates", "test_candidates",
]

for split in splits:
    data = {"candidate_data": {}}
    for row in dataset[split]:
        data["candidate_data"][row["id"]] = row["text"]

    save_path = os.path.join(rr_input_path, f"{split}.sav")
    with open(save_path, "wb") as f:
        pkl.dump(data, f)
    print(f"Saved {split:25s} → {save_path}  ({len(data['candidate_data'])} docs)")

Saved train_queries             → ../others/Rhetorical-Roles/Code/RR_pipeline/Input_Data/train_queries.sav  (827 docs)
Saved dev_queries               → ../others/Rhetorical-Roles/Code/RR_pipeline/Input_Data/dev_queries.sav  (118 docs)
Saved test_queries              → ../others/Rhetorical-Roles/Code/RR_pipeline/Input_Data/test_queries.sav  (237 docs)
Saved train_candidates          → ../others/Rhetorical-Roles/Code/RR_pipeline/Input_Data/train_candidates.sav  (4320 docs)
Saved dev_candidates            → ../others/Rhetorical-Roles/Code/RR_pipeline/Input_Data/dev_candidates.sav  (1023 docs)
Saved test_candidates           → ../others/Rhetorical-Roles/Code/RR_pipeline/Input_Data/test_candidates.sav  (1727 docs)


### Run the RR Pipeline

The `.sav` files are now ready. Run `rr_pipeline.py` from your terminal (requires the model weights and a CUDA GPU):

```bash
cd ../others/Rhetorical-Roles/Code/RR_pipeline
python rr_pipeline.py
```

The pipeline will generate per-split RR results under `Output_Data/RR_Output_Data/` as `<split>_rr_results.json`.  
Each file has the format:
```json
{
  "case_id": [["sentence text", "RR_Label"], ...]
}
```

Once all 6 splits are processed, continue with the cells below.

In [4]:
import json

# Load all RR results from the pipeline output into a flat lookup dict.
# rr_labels[case_id] = [["sentence text", "RR_Label"], ...]
rr_output_path = os.path.join(RR_PIPELINE_DIR, "Output_Data", "RR_Output_Data")
rr_labels = {}

for split in splits:
    result_file = os.path.join(rr_output_path, f"{split}_rr_results.json")
    if os.path.exists(result_file):
        with open(result_file, "r") as f:
            results = json.load(f)
        rr_labels.update(results)
        print(f"Loaded {split:25s}: {len(results)} docs")
    else:
        print(f"[NOT FOUND] {result_file}  ← run rr_pipeline.py first")

print(f"\nTotal docs with RR labels: {len(rr_labels)}")

[NOT FOUND] ../others/Rhetorical-Roles/Code/RR_pipeline/Output_Data/RR_Output_Data/train_queries_rr_results.json  ← run rr_pipeline.py first
[NOT FOUND] ../others/Rhetorical-Roles/Code/RR_pipeline/Output_Data/RR_Output_Data/dev_queries_rr_results.json  ← run rr_pipeline.py first
[NOT FOUND] ../others/Rhetorical-Roles/Code/RR_pipeline/Output_Data/RR_Output_Data/test_queries_rr_results.json  ← run rr_pipeline.py first
[NOT FOUND] ../others/Rhetorical-Roles/Code/RR_pipeline/Output_Data/RR_Output_Data/train_candidates_rr_results.json  ← run rr_pipeline.py first
[NOT FOUND] ../others/Rhetorical-Roles/Code/RR_pipeline/Output_Data/RR_Output_Data/dev_candidates_rr_results.json  ← run rr_pipeline.py first
[NOT FOUND] ../others/Rhetorical-Roles/Code/RR_pipeline/Output_Data/RR_Output_Data/test_candidates_rr_results.json  ← run rr_pipeline.py first

Total docs with RR labels: 0


In [ ]:
# RR labels ordered by relevance to prior case retrieval.
# RatioOfTheDecision = core legal reasoning (ratio decidendi) → most distinctive across cases
# Statute            = specific laws cited → cases sharing statutes are likely related
# Precedent          = prior cases cited → directly points at relevant candidates
# Argument           = type of legal dispute → stylistically stable across similar cases
HIGH_VALUE_RR_LABELS = {"RatioOfTheDecision", "Statute", "Precedent", "Argument"}


def get_rr_filtered_text(case_id: str, fallback_sentences: list) -> str:
    """
    Returns text consisting only of high-value RR sentences.
    Falls back to joining all sentences if no RR labels are available,
    or if none of the sentences carry a high-value label.
    """
    if case_id not in rr_labels:
        return " ".join(fallback_sentences)

    pairs = rr_labels[case_id]  # list of [sentence, label]
    filtered = [sent for sent, label in pairs if label in HIGH_VALUE_RR_LABELS]
    return " ".join(filtered) if filtered else " ".join(fallback_sentences)


def get_rr_sentences_by_label(case_id: str) -> dict:
    """
    Returns a dict mapping each RR label → list of sentences for that label.
    Used by the advanced stratified similarity approach.
    """
    if case_id not in rr_labels:
        return {}
    result = {}
    for sent, label in rr_labels[case_id]:
        result.setdefault(label, []).append(sent)
    return result


# Quick sanity check
sample_id = dataset["train_queries"][0]["id"]
filtered = get_rr_filtered_text(sample_id, dataset["train_queries"][0]["text"])
print(f"Case ID:           {sample_id}")
print(f"Original sentences: {len(dataset['train_queries'][0]['text'])}")
print(f"Filtered text preview:\n{filtered[:500]}...")

In [4]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME  = "nlpaueb/legal-bert-base-uncased"  # swap to "bert-base-uncased" if desired
USE_RR      = False  # set True once rr_labels is populated by the RR pipeline
BATCH_SIZE  = 16     # ↑ from 2 — larger batches = better GPU utilisation + more in-batch negatives for MNRL
USE_AMP     = True   # FP16 mixed-precision (~1.5× speedup on CUDA)
# ─────────────────────────────────────────────────────────────────────────────

model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = 512

train_cands_dict = {row["id"]: row for row in dataset["train_candidates"]}

train_examples = []
n_rr, n_fallback = 0, 0

for row in dataset["train_queries"]:
    qid = row["id"]

    if USE_RR and rr_labels:
        query_text = get_rr_filtered_text(qid, row["text"])
        if qid in rr_labels:
            n_rr += 1
        else:
            n_fallback += 1
    else:
        query_text = " ".join(row["text"])

    for cand_id in row["relevant_candidates"]:
        if cand_id not in train_cands_dict:
            continue
        cand_row = train_cands_dict[cand_id]

        if USE_RR and rr_labels:
            cand_text = get_rr_filtered_text(cand_id, cand_row["text"])
        else:
            cand_text = " ".join(cand_row["text"])

        train_examples.append(InputExample(texts=[query_text, cand_text]))

mode_str = f"RR-filtered ({n_rr} RR, {n_fallback} fallback)" if USE_RR else "full-text (no RR)"
print(f"Mode: {mode_str}")
print(f"Training pairs: {len(train_examples)}")

output_path = "./pcr_rr_biencoder_model" if USE_RR else "./pcr_biencoder_model"

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    show_progress_bar=True,
    use_amp=USE_AMP,
    output_path=output_path,
)


No sentence-transformers model found with name nlpaueb/legal-bert-base-uncased. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mode: full-text (no RR)
Training pairs: 5308


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 10.75 MiB is free. Including non-PyTorch memory, this process has 3.60 GiB memory in use. Of the allocated memory 3.39 GiB is allocated by PyTorch, and 115.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Advanced: RR-Stratified Similarity Scoring

Instead of filtering to a single text, we can compute **per-label MaxSim** scores and combine them.  
For each query-candidate pair:

$$\text{sim}(q, c) = \sum_{L \in \text{HIGH\_VALUE}} w_L \cdot \max_{s_q \in q_L,\, s_c \in c_L} \cos(e(s_q),\, e(s_c))$$

Where $q_L$ and $c_L$ are the sentences with label $L$ in the query and candidate respectively.  
This is inspired by U-CREAT's event-IOU approach, but uses dense sentence embeddings instead of Jaccard over event triples.

**Label weights** reflect retrieval importance — `RatioOfTheDecision` > `Statute` > `Precedent` > `Argument`.

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Importance weights for each RR label
RR_WEIGHTS = {
    "RatioOfTheDecision": 0.40,
    "Statute":            0.25,
    "Precedent":          0.20,
    "Argument":           0.15,
}


def encode_sentences(sentence_list: list, enc_model: SentenceTransformer) -> torch.Tensor:
    """Encode a list of sentences → tensor[N, emb_dim]."""
    if not sentence_list:
        return None
    embs = enc_model.encode(sentence_list, convert_to_tensor=True, show_progress_bar=False)
    return F.normalize(embs, dim=-1)


def _build_label_embs(case_id: str, fallback_sentences: list, enc_model: SentenceTransformer) -> dict:
    """
    Returns {label: tensor} for all high-value labels present in a case.
    Uses special key "_full" when no RR labels are available (fallback).
    """
    by_label = get_rr_sentences_by_label(case_id)
    if not by_label:
        return {"_full": encode_sentences([get_rr_filtered_text(case_id, fallback_sentences)], enc_model)}
    result = {}
    for label in RR_WEIGHTS:
        sents = by_label.get(label, [])
        if sents:
            result[label] = encode_sentences(sents, enc_model)
    return result or {"_full": encode_sentences([get_rr_filtered_text(case_id, fallback_sentences)], enc_model)}


def _score_from_embs(q_embs: dict, c_embs: dict) -> float:
    """Compute weighted MaxSim score given pre-computed embedding dicts."""
    # Fallback: either side has no RR labels
    if "_full" in q_embs or "_full" in c_embs:
        q_e = q_embs.get("_full") or next(iter(q_embs.values()))
        c_e = c_embs.get("_full") or next(iter(c_embs.values()))
        return float(torch.matmul(q_e, c_e.T).squeeze())

    total_score, total_weight = 0.0, 0.0
    for label, weight in RR_WEIGHTS.items():
        q_e = q_embs.get(label)
        c_e = c_embs.get(label)
        if q_e is None or c_e is None:
            continue
        max_sim = float(torch.matmul(q_e, c_e.T).max())
        total_score += weight * max_sim
        total_weight += weight
    return total_score / total_weight if total_weight > 0 else 0.0


def evaluate_rr_stratified(enc_model, dataset_queries, dataset_candidates, k=50):
    """
    Evaluation loop using RR-stratified similarity.
    Candidate embeddings are pre-computed ONCE (not per query) for a major speedup.
    """
    cand_rows = {row["id"]: row for row in dataset_candidates}
    cand_ids  = list(cand_rows.keys())
    ground_truth = {row["id"]: set(row["relevant_candidates"]) for row in dataset_queries}

    # ── Pre-encode all candidates once ──────────────────────────────────────
    cand_embs = {}
    for cid in tqdm(cand_ids, desc="Pre-encoding candidates"):
        cand_embs[cid] = _build_label_embs(cid, cand_rows[cid]["text"], enc_model)

    # ── Query loop ───────────────────────────────────────────────────────────
    TP, FP, FN = 0, 0, 0

    for q_row in tqdm(dataset_queries, desc="Queries"):
        qid    = q_row["id"]
        q_embs = _build_label_embs(qid, q_row["text"], enc_model)

        scores = {cid: _score_from_embs(q_embs, cand_embs[cid]) for cid in cand_ids}

        top_k_cands = set(sorted(scores, key=scores.get, reverse=True)[:k])
        relevant    = ground_truth[qid]

        tp   = len(top_k_cands & relevant)
        TP  += tp
        FP  += len(top_k_cands) - tp
        FN  += len(relevant) - tp

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    micro_f1  = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return micro_f1, precision, recall


SAMPLE_SIZE = 20
dev_queries_sample = dataset["dev_queries"].select(range(SAMPLE_SIZE))

mu_f1, p, r = evaluate_rr_stratified(
    model, dev_queries_sample, dataset["dev_candidates"], k=7
)
print(f"\nRR-Stratified Evaluation @ K=7  (sample={SAMPLE_SIZE} queries)")
print(f"Micro F1:  {mu_f1:.4f}")
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
